In [1]:
# from PIL import Image, PngImagePlugin
# from pathlib import Path

# PngImagePlugin.MAX_TEXT_CHUNK = 100 * (1024**2)  # raise limit so we can open them

# folder = Path("/figures/laion_5k_generated_pt")

# for img_path in folder.glob("*.png"):
#     try:
#         img = Image.open(img_path).convert("RGB")
#         img.save(img_path, format="PNG", icc_profile=None)
#     except Exception as e:
#         print(f"Failed {img_path.name}: {e}")

# print("Done")

In [2]:
# from datasets import load_dataset
# import pandas as pd

# # LAION Aesthetics V2 (most commonly cited)
# ds = load_dataset("laion/aesthetics_v2_4.5", split="train", streaming=True)

# # Take first 5000 entries
# subset = []
# for i, item in enumerate(ds):
#     if i >= 5000:
#         break
#     subset.append({
#         "url": item["URL"],
#         "text": item["TEXT"],       # ← this is your text prompt
#     })

# df = pd.DataFrame(subset)
# df.to_csv("laion_5k_prompts.csv", index=False)
# print(f"Saved {len(df)} rows")
# print(df.head())

In [3]:
# import requests
# from pathlib import Path
# import pandas as pd
# from PIL import Image
# from io import BytesIO

# real_dir = Path("/n/netscratch/kempner_undergrads/Everyone/zwu/parallel_toy/figures/laion_5k_real")
# real_dir.mkdir(exist_ok=True)

# df = pd.read_csv("laion_5k_prompts.csv")
# failed = 0

# for idx, row in df.iterrows():
#     img_path = real_dir / f"{idx:05d}.png"
#     if img_path.exists():
#         continue  # resume if interrupted
#     try:
#         r = requests.get(row["url"], timeout=5)
#         r.raise_for_status()
#         # Convert to PNG via PIL to avoid saving corrupt/non-image bytes
#         img = Image.open(BytesIO(r.content)).convert("RGB")
#         img.save(img_path)
#     except Exception as e:
#         failed += 1

# print(f"Done. {len(df) - failed}/{len(df)} images saved, {failed} failed.")

In [ ]:
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image, PngImagePlugin
from tqdm import tqdm
from transformers import CLIPModel, CLIPProcessor
import cleanfid.fid as fid
from cleanfid.fid import get_folder_features, frechet_distance

# ── Fix PIL ICC profile limit ─────────────────────────────────────────────────
PngImagePlugin.MAX_TEXT_CHUNK = 100 * (1024**2)

# ── Config ────────────────────────────────────────────────────────────────────
REAL_DIR    = "/n/netscratch/kempner_undergrads/Everyone/zwu/parallel_toy/figures/laion_5k_real"
TSR_DIR     = Path("/n/netscratch/kempner_undergrads/Everyone/zwu/parallel_toy/figures/laion_5k_generated")
PT_TSR_DIR  = Path("/n/netscratch/kempner_undergrads/Everyone/zwu/parallel_toy/figures/laion_5k_generated_pt")
PROMPTS_FILE = "laion_5k_prompts.csv"
K_VALUES    = [1.0, 0.98, 0.95, 0.93, 0.9]

df = pd.read_csv(PROMPTS_FILE)
prompts = df["text"].tolist()[:100]

# ── CLIP setup ────────────────────────────────────────────────────────────────
device = "cuda" if torch.cuda.is_available() else "cpu"
clip_model     = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(device)
clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

# ── FID: precompute real features once ───────────────────────────────────────
print("Computing real image features (one time)...")
feat_model = fid.build_feature_extractor("clean", device=torch.device(device))
real_feats = get_folder_features(REAL_DIR, feat_model, device=torch.device(device))
mu_real    = np.mean(real_feats, axis=0)
sigma_real = np.cov(real_feats, rowvar=False)
print(f"Real features computed from {len(real_feats)} images.\n")

# ── Helpers ───────────────────────────────────────────────────────────────────
def clean_images(gen_dir, prefix):
    """Copy images to tmp dir with ICC profile stripped."""
    gen_dir  = Path(gen_dir)
    tmp_dir  = Path("/tmp/fid_clean") / f"{prefix}_{gen_dir.name}"
    tmp_dir.mkdir(parents=True, exist_ok=True)

    for img_path in gen_dir.glob("*.png"):
        out_path = tmp_dir / img_path.name
        if not out_path.exists():
            try:
                img = Image.open(img_path).convert("RGB")
                img.save(out_path, format="PNG", icc_profile=None)
            except Exception as e:
                print(f"  Warning, skipping {img_path.name}: {e}")

    n = len(list(tmp_dir.glob("*.png")))
    print(f"  [clean] {prefix}_{gen_dir.name}: {n} images")
    return tmp_dir

def compute_fid_score(gen_dir, prefix=""):
    tmp_dir   = clean_images(gen_dir, prefix)
    gen_feats = get_folder_features(str(tmp_dir), feat_model, device=torch.device(device))
    mu_gen    = np.mean(gen_feats, axis=0)
    sigma_gen = np.cov(gen_feats, rowvar=False)
    return frechet_distance(mu_real, sigma_real, mu_gen, sigma_gen)

def compute_clip(gen_dir):
    scores = []
    for idx, prompt in enumerate(tqdm(prompts, desc=f"CLIP {gen_dir.name}")):
        img_path = gen_dir / f"{idx:05d}.png"
        if not img_path.exists():
            continue
        image  = Image.open(img_path).convert("RGB")
        inputs = clip_processor(
            text=[prompt], images=image,
            return_tensors="pt", padding=True,
            truncation=True, max_length=75
        ).to(device)
        with torch.no_grad():
            score = clip_model(**inputs).logits_per_image[0, 0].item()
        scores.append(score)
    return np.mean(scores)

# ── Compute for both TSR and PT-TSR ──────────────────────────────────────────
tsr_results    = {}
pt_tsr_results = {}

for tsr_k in K_VALUES:
    k_str = f"k{tsr_k:.3f}".replace(".", "p")
    print(f"\n── {k_str} ──")

    # TSR
    tsr_gen_dir = TSR_DIR / k_str
    tsr_fid_val  = compute_fid_score(tsr_gen_dir, prefix="tsr")
    tsr_clip_val = compute_clip(tsr_gen_dir)
    tsr_results[tsr_k] = (tsr_fid_val, tsr_clip_val)
    print(f"[TSR]    k={tsr_k:.3f}  FID={tsr_fid_val:.4f}  CLIP={tsr_clip_val:.4f}")

    # PT-TSR
    pt_gen_dir = PT_TSR_DIR / k_str
    pt_fid_val  = compute_fid_score(pt_gen_dir, prefix="pt")
    pt_clip_val = compute_clip(pt_gen_dir)
    pt_tsr_results[tsr_k] = (pt_fid_val, pt_clip_val)
    print(f"[PT-TSR] k={tsr_k:.3f}  FID={pt_fid_val:.4f}  CLIP={pt_clip_val:.4f}")

# ── Summary ───────────────────────────────────────────────────────────────────
print("\n── FID (lower is better) ──")
for k in K_VALUES:
    print(f"  k={k:.3f}  TSR={tsr_results[k][0]:.4f}  PT-TSR={pt_tsr_results[k][0]:.4f}")

print("\n── CLIP (higher is better) ──")
for k in K_VALUES:
    print(f"  k={k:.3f}  TSR={tsr_results[k][1]:.4f}  PT-TSR={pt_tsr_results[k][1]:.4f}")

# ── Plot ──────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 6))

tsr_k_vals    = sorted(tsr_results.keys(), reverse=True)
tsr_clip_vals = [tsr_results[k][1] for k in tsr_k_vals]
tsr_fid_vals  = [tsr_results[k][0] for k in tsr_k_vals]
ax.plot(tsr_clip_vals, tsr_fid_vals, color="gold", marker="o", linewidth=2, label="TSR, CFG=7.5, σ=3.0")
for k in tsr_k_vals:
    f, c = tsr_results[k]
    ax.annotate(f"k={k}", (c, f), textcoords="offset points", xytext=(6, 0), fontsize=8, color="goldenrod")

pt_k_vals    = sorted(pt_tsr_results.keys(), reverse=True)
pt_clip_vals = [pt_tsr_results[k][1] for k in pt_k_vals]
pt_fid_vals  = [pt_tsr_results[k][0] for k in pt_k_vals]
ax.plot(pt_clip_vals, pt_fid_vals, color="orangered", marker="o", linewidth=2, label="PT-TSR, CFG=7.5, σ=3.0")
for k in pt_k_vals:
    f, c = pt_tsr_results[k]
    ax.annotate(f"k={k}", (c, f), textcoords="offset points", xytext=(6, 0), fontsize=8, color="orangered")

ax.set_xlabel("CLIP", fontsize=12)
ax.set_ylabel("FID", fontsize=12)
ax.set_title("FID vs CLIP comparison", fontsize=14)
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("fid_vs_clip.png", dpi=150)
plt.show()
print("Saved fid_vs_clip.png")

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Computing real image features (one time)...


/n/home00/zoewu/.local/lib/python3.12/site-packages/torch/utils/data/dataloader.py:624: UserWarning: This DataLoader will create 12 worker processes in total. Our suggested max number of worker in current system is 1, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(


Found 3025 images in the folder /n/netscratch/kempner_undergrads/Everyone/zwu/parallel_toy/figures/laion_5k_real


100%|██████████| 24/24 [01:20<00:00,  3.37s/it]


...

CLIP k0p980: 100%|██████████| 100/100 [00:05<00:00, 19.35it/s]


[PT-TSR] k=0.980  FID=191.3370  CLIP=32.4365

── k0p950 ──
  [clean] tsr_k0p950: 100 images
Found 100 images in the folder /tmp/fid_clean/tsr_k0p950


CLIP k0p950: 100%|██████████| 100/100 [00:05<00:00, 19.57it/s]


[TSR]    k=0.950  FID=190.1160  CLIP=32.4533
  [clean] pt_k0p950: 100 images
Found 100 images in the folder /tmp/fid_clean/pt_k0p950


  0%|          | 0/1 [00:00<?, ?it/s]